# 13_07_致命的な組み合わせを構造で断つ

コード：`ch13/blocks/13_07_致命的な組み合わせを構造で断つ.ipynb`

本文の断片は参照表示し、末尾のセルで前提を補う既存ランナーを実行します。関数・変数・外部処理が省略されているため、断片だけを単独実行しないでください。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import os
import pathlib
import sys
import zipfile

CODE = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents,
                        pathlib.Path.cwd() / 'gradient-attention-agent']
             if (p / 'run_all.py').is_file() and (p / 'common/blocks.py').is_file()), None)
if CODE is None:
    from google.colab import files
    uploaded = files.upload()  # gradient-attention-agent-code.zip を選ぶ
    if len(uploaded) != 1:
        raise ValueError('配布ZIPを1つ選んでください')
    dest = pathlib.Path.cwd().resolve()
    with zipfile.ZipFile(next(iter(uploaded))) as z:
        for member in z.namelist():
            if not (dest / member).resolve().is_relative_to(dest / 'gradient-attention-agent'):
                raise ValueError('同梱のコード配布ZIPを選んでください')
        z.extractall(dest)
    CODE = dest / 'gradient-attention-agent'
    del uploaded
print('コード一式:', CODE)


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2']], check=True)


## 本文の掲載コード（参照）


```python
from enum import Flag, auto

class Cap(Flag):
    NONE       = 0
    UNTRUSTED  = auto()   # Web・メール・外部文書など、他人が書いた内容を読む
    SENSITIVE  = auto()   # 顧客DB・社内文書・認証情報に触れる
    EXFIL      = auto()   # 外部への送信ができる（メール・HTTP・投稿）

CAPS = {
    "fetch_url":      Cap.UNTRUSTED | Cap.EXFIL,   # URL を叩く＝送信でもある
    "read_email":     Cap.UNTRUSTED,
    "search_customer_db": Cap.SENSITIVE,
    "send_email":     Cap.EXFIL,
    "calculator":     Cap.NONE,
}
TRIFECTA = Cap.UNTRUSTED | Cap.SENSITIVE | Cap.EXFIL

def build_agent(tool_names):
    total = Cap.NONE
    for t in tool_names:
        total |= CAPS[t]
    if total & TRIFECTA == TRIFECTA:              # ★ 3つ揃ったら起動させない
        raise ValueError(
            f"lethal trifecta: {tool_names} は "
            "「外部入力・機密・送信」を同時に持ちます。エージェントを分割してください。")
    return Agent(tool_names)

# OK : 外部を読んで要約するだけ（機密に触れない）
build_agent(["fetch_url", "calculator"])
# NG : 外部を読み、顧客DBを引き、メールを送れる
build_agent(["fetch_url", "search_customer_db", "send_email"])   # → ValueError
```


## 前提を補って実行する

既存ランナーが関連する断片と必要なデータ・スタブを接続します。外部サービスを実行した証明や、掲載用AI応答の完全証跡にはなりません。


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, str(CODE / "run_all.py"), "--only", "13", "--name=agent", "--skip-docker"], cwd=CODE, check=True)
